<!-- DOC-BANNER -->
# Working with JSON in Pandas — Deep Dive (Day 16 extension)

> **Playlist:** CampusX 100 Days of ML — Day 16 *Working with JSON/SQL* (extended practice; this notebook goes deeper than the video).
> **Purpose:** Master `pd.read_json` orientations, `to_json`, JSON Lines, `json` module, `pd.json_normalize` (incl. `record_path`, `meta`, `sep`, `max_level`), `explode`, and dates/unicode — using a `students.json` you create inline.
> **Prerequisites:** Python dicts/lists, Pandas DataFrame basics, Day 15 CSV reading.
> **After studying:** you can decide tabular→`read_json`, nested→`json_normalize`, and handle StringIO/JSONL/date_unit pitfalls (incl. the intentional `FileNotFoundError` demo for passing a JSON *string* without `StringIO`).


# Working with JSON Files in Pandas — Complete Notes

A practical, beginner-friendly notebook covering the main JSON workflows you will use with **Pandas**.

### What this notebook covers
1. What JSON is and its common structures
2. `read_json()` — reading JSON files
3. `to_json()` — writing DataFrames to JSON
4. Important JSON orientations (`records`, `columns`, `index`, `split`, `table`, `values`)
5. Reading JSON from strings
6. Nested JSON and `json_normalize()`
7. `record_path`, `meta`, and `meta_prefix`
8. Flattening nested dictionaries
9. JSON columns inside a DataFrame
10. Converting JSON strings with `json.loads()` / `json.dumps()`
11. Handling missing/null values
12. Date handling
13. Common parameters and use cases
14. Practical examples and a mini cheat sheet

> **Goal:** Understand *which function to use, why to use it, and what output it produces*.


## 1. JSON basics

JSON (JavaScript Object Notation) is a common format for exchanging and storing structured data.

Typical JSON structures:

- **Object / dictionary** → `{ "name": "Amit", "age": 22 }`
- **Array / list** → `[{"name":"Amit"}, {"name":"Rahul"}]`
- **Nested object** → an object containing another object
- **Nested array** → an object containing a list of objects

Pandas is especially useful when JSON contains tabular or semi-tabular data.


In [1]:
import pandas as pd
import json
from pathlib import Path

pd.__version__


'3.0.5'

## 2. Create sample JSON data

We will use a small dataset throughout the notebook.


In [2]:
data = [
    {
        "id": 1,
        "name": "Amit",
        "age": 22,
        "city": "Noida",
        "skills": ["Python", "SQL"],
        "address": {"state": "UP", "pincode": 201301}
    },
    {
        "id": 2,
        "name": "Rahul",
        "age": 23,
        "city": "Delhi",
        "skills": ["Python", "ML"],
        "address": {"state": "Delhi", "pincode": 110001}
    },
    {
        "id": 3,
        "name": "Priya",
        "age": 21,
        "city": "Lucknow",
        "skills": ["SQL", "Tableau"],
        "address": {"state": "UP", "pincode": 226001}
    }
]

# Save as a JSON file
with open("students.json", "w", encoding="utf-8") as f:
    json.dump(data, f, indent=4)

print("students.json created")


students.json created


# 3. `pd.read_json()` — Read JSON into a DataFrame

### Basic syntax

```python
pd.read_json(path_or_buffer, orient=None, typ='frame', lines=False, ...)
```

### Main use cases

| Situation | Use |
|---|---|
| Normal JSON file | `pd.read_json("file.json")` |
| JSON Lines / NDJSON | `pd.read_json("file.json", lines=True)` |
| JSON string | `pd.read_json(StringIO(json_string))` |
| Specific orientation | `pd.read_json(..., orient="records")` |

**Important:** The JSON structure/orientation determines how Pandas interprets the data.


In [3]:
df = pd.read_json("students.json")
df


,id,name,age,city,skills,address
0,1,Amit,22,Noida,"[Python, SQL]","{'state': 'UP', 'pincode': 201301}"
1,2,Rahul,23,Delhi,"[Python, ML]","{'state': 'Delhi', 'pincode': 110001}"
2,3,Priya,21,Lucknow,"[SQL, Tableau]","{'state': 'UP', 'pincode': 226001}"


## 4. `orient` — the most important JSON concept

JSON can represent the same DataFrame in different formats.

Common orientations:

- `records`
- `columns`
- `index`
- `split`
- `table`
- `values`

### `records`

Each row becomes one JSON object.

```json
[
  {"name": "Amit", "age": 22},
  {"name": "Rahul", "age": 23}
]
```

Best for APIs and exchanging rows.


In [ ]:
df_small = pd.DataFrame({
    "name": ["Amit", "Rahul"],
    "age": [22, 23],
    "city": ["Noida", "Delhi"]
})

records_json = df_small.to_json(orient="records", indent=2)
print(records_json)

pd.read_json(records_json, orient="records")


[
  {
    "name":"Amit",
    "age":22,
    "city":"Noida"
  },
  {
    "name":"Rahul",
    "age":23,
    "city":"Delhi"
  }
]


FileNotFoundError: File [
  {
    "name":"Amit",
    "age":22,
    "city":"Noida"
  },
  {
    "name":"Rahul",
    "age":23,
    "city":"Delhi"
  }
] does not exist

: 

### `columns`

Each column is represented as a dictionary:

```json
{
  "name": {"0": "Amit", "1": "Rahul"},
  "age": {"0": 22, "1": 23}
}
```

This is the default orientation used by `DataFrame.to_json()`.

**Use case:** preserving a DataFrame-like column/index structure.


In [ ]:
columns_json = df_small.to_json(orient="columns", indent=2)
print(columns_json)


### `index`

The outer keys are index values.

**Use case:** when the DataFrame index is important.


In [ ]:
index_json = df_small.to_json(orient="index", indent=2)
print(index_json)


### `split`

Stores the DataFrame as:

- `columns`
- `index`
- `data`

**Use case:** compactly representing the complete DataFrame structure.


In [ ]:
split_json = df_small.to_json(orient="split", indent=2)
print(split_json)


### `table`

Stores schema information plus the data.

**Use case:** more self-describing JSON, especially when schema/type information matters.


In [ ]:
table_json = df_small.to_json(orient="table", indent=2)
print(table_json)


### `values`

Only the values are stored; column names and index are lost.

**Use case:** when you only need the raw matrix of values.


In [ ]:
values_json = df_small.to_json(orient="values")
print(values_json)


## 5. `DataFrame.to_json()` — write DataFrame to JSON

### Basic use

```python
df.to_json("output.json")
```

### Useful parameters

- `path_or_buf` → output file or `None`
- `orient` → JSON structure
- `indent` → pretty formatting
- `date_format` → date representation
- `date_unit` → `s`, `ms`, `us`, `ns`
- `lines=True` → one JSON object per line
- `force_ascii=False` → preserve Unicode characters


In [ ]:
df_small.to_json("students_records.json", orient="records", indent=2)

df_small.to_json(
    "students_pretty.json",
    orient="records",
    indent=4,
    force_ascii=False
)

print("JSON files written.")


# 6. JSON Lines / NDJSON

JSON Lines means **one JSON object per line**:

```text
{"id": 1, "name": "Amit"}
{"id": 2, "name": "Rahul"}
{"id": 3, "name": "Priya"}
```

### Read

```python
pd.read_json("file.json", lines=True)
```

### Write

```python
df.to_json("file.json", orient="records", lines=True)
```

**Use case:** large datasets, logs, streaming data, APIs.


In [ ]:
df_small.to_json("students.jsonl", orient="records", lines=True)

jsonl_df = pd.read_json("students.jsonl", lines=True)
jsonl_df


# 7. Reading a JSON string

For JSON strings, `StringIO` is useful.

Modern Pandas versions may warn when passing literal JSON directly to `read_json()`. Using `StringIO` is explicit and reliable.

```python
from io import StringIO
pd.read_json(StringIO(json_string), orient="records")
```


In [ ]:
from io import StringIO

json_string = '[{"name":"Amit","age":22},{"name":"Rahul","age":23}]'

df_from_string = pd.read_json(
    StringIO(json_string),
    orient="records"
)

df_from_string


# 8. Python `json` module vs Pandas

Pandas and Python's built-in `json` module solve slightly different problems.

### `json`
Use when you need to work with JSON as Python dictionaries/lists.

- `json.load()` → JSON file → Python object
- `json.loads()` → JSON string → Python object
- `json.dump()` → Python object → JSON file
- `json.dumps()` → Python object → JSON string

### Pandas
Use when the JSON represents tabular data.

- `pd.read_json()` → JSON → DataFrame
- `df.to_json()` → DataFrame → JSON
- `pd.json_normalize()` → nested JSON → DataFrame


In [ ]:
# json.loads: JSON string -> Python object
json_text = '{"name": "Amit", "age": 22}'
python_object = json.loads(json_text)

print(python_object)
print(type(python_object))

# json.dumps: Python object -> JSON string
back_to_json = json.dumps(python_object, indent=2)
print(back_to_json)


# 9. `pd.json_normalize()` — nested JSON

This is one of the **most important functions** when working with real-world APIs.

Suppose JSON contains nested dictionaries:

```python
{
    "name": "Amit",
    "address": {
        "city": "Noida",
        "state": "UP"
    }
}
```

`json_normalize()` flattens nested objects into columns.


In [ ]:
nested_data = [
    {
        "id": 1,
        "name": "Amit",
        "address": {"city": "Noida", "state": "UP"},
        "education": {"degree": "BTech", "year": 2027}
    },
    {
        "id": 2,
        "name": "Rahul",
        "address": {"city": "Delhi", "state": "Delhi"},
        "education": {"degree": "BTech", "year": 2026}
    }
]

flat_df = pd.json_normalize(nested_data)
flat_df


## 10. `sep` in `json_normalize()`

By default, nested columns are named like:

```text
address.city
address.state
```

You can change the separator.

```python
pd.json_normalize(data, sep="_")
```

Result:

```text
address_city
address_state
```


In [ ]:
pd.json_normalize(nested_data, sep="_")


# 11. `max_level`

`max_level` controls how deeply dictionaries are normalized.

### `max_level=0`
Keep nested dictionaries instead of flattening them.

### `max_level=1`
Flatten one level.

**Use case:** useful when you do not want to completely flatten a complicated API response.


In [ ]:
print("max_level=0")
display(pd.json_normalize(nested_data, max_level=0))

print("max_level=1")
display(pd.json_normalize(nested_data, max_level=1))


# 12. Nested arrays and `record_path`

This is extremely important for API JSON.

Example:

```python
{
    "student": "Amit",
    "courses": [
        {"name": "Python", "score": 90},
        {"name": "ML", "score": 85}
    ]
}
```

If `courses` is the repeated record you want as rows, use:

```python
pd.json_normalize(data, record_path="courses")
```


In [ ]:
course_data = [
    {
        "student": "Amit",
        "college": "GL Bajaj",
        "courses": [
            {"name": "Python", "score": 90},
            {"name": "ML", "score": 85}
        ]
    },
    {
        "student": "Rahul",
        "college": "GL Bajaj",
        "courses": [
            {"name": "Python", "score": 88},
            {"name": "SQL", "score": 92}
        ]
    }
]

courses_df = pd.json_normalize(
    course_data,
    record_path="courses"
)

courses_df


# 13. `meta` with `record_path`

The problem with only `record_path` is that parent information such as `student` can be lost.

Use `meta` to bring parent fields into every child row.

```python
pd.json_normalize(
    data,
    record_path="courses",
    meta=["student", "college"]
)
```


In [ ]:
courses_with_meta = pd.json_normalize(
    course_data,
    record_path="courses",
    meta=["student", "college"]
)

courses_with_meta


## 14. `meta_prefix`

If parent metadata names could conflict with child fields, add a prefix.

```python
meta_prefix="student_"
```


In [ ]:
pd.json_normalize(
    course_data,
    record_path="courses",
    meta=["student", "college"],
    meta_prefix="parent_"
)


# 15. Nested `record_path`

`record_path` can also point to nested lists.

Example:

```python
record_path=["department", "employees"]
```

Use this when the list you want is several levels deep.


In [ ]:
deep_data = [
    {
        "company": "ABC",
        "department": {
            "name": "AI",
            "employees": [
                {"name": "Amit", "role": "Intern"},
                {"name": "Rahul", "role": "Engineer"}
            ]
        }
    }
]

pd.json_normalize(
    deep_data,
    record_path=["department", "employees"],
    meta=["company", ["department", "name"]]
)


# 16. JSON stored inside a DataFrame column

A very common real-world situation:

```text
id | details
1  | '{"city":"Noida","age":22}'
2  | '{"city":"Delhi","age":23}'
```

The `details` column contains **JSON strings**, not dictionaries.

You first parse them using `json.loads()`, then normalize them.


In [ ]:
df_json_column = pd.DataFrame({
    "id": [1, 2],
    "details": [
        '{"city":"Noida","age":22}',
        '{"city":"Delhi","age":23}'
    ]
})

df_json_column["details"] = df_json_column["details"].apply(json.loads)

details_flat = pd.json_normalize(df_json_column["details"])
result = pd.concat(
    [df_json_column.drop(columns="details").reset_index(drop=True),
     details_flat.reset_index(drop=True)],
    axis=1
)

result


# 17. JSON column containing lists

If a column contains lists, `explode()` is often useful.

Example:

```text
id | skills
1  | [Python, SQL]
2  | [ML, Python]
```

Use:

```python
df.explode("skills")
```

This is not a JSON-specific function, but it is very useful after loading JSON data.


In [ ]:
skills_df = pd.DataFrame({
    "id": [1, 2],
    "skills": [["Python", "SQL"], ["ML", "Python"]]
})

skills_df.explode("skills")


# 18. JSON null vs Python/Pandas missing values

JSON uses:

```json
null
```

Python uses:

```python
None
```

Pandas generally represents missing numeric values using `NaN` or nullable dtypes.

Example:


In [ ]:
missing_json = '[{"name":"Amit","age":22},{"name":"Rahul","age":null}]'

missing_df = pd.read_json(
    StringIO(missing_json),
    orient="records"
)

missing_df


# 19. Checking and cleaning JSON-loaded data

After loading JSON, normal Pandas functions become available:

```python
df.head()
df.info()
df.shape
df.columns
df.dtypes
df.isna()
df.dropna()
df.fillna()
df.astype()
df.rename()
df.query()
df.groupby()
df.sort_values()
```

JSON loading is only the beginning — after conversion, treat the result like a normal DataFrame.


In [ ]:
df = pd.read_json("students.json")

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nDtypes:")
print(df.dtypes)

display(df.head())


# 20. Dates in JSON

JSON does not have a native date type in the same way Python/Pandas does.

You may receive dates as strings:

```json
{"date": "2026-09-23"}
```

Convert them after loading:

```python
df["date"] = pd.to_datetime(df["date"])
```

For writing JSON, `date_format` can control how dates are serialized.


In [ ]:
dates_df = pd.DataFrame({
    "name": ["Amit", "Rahul"],
    "date": pd.to_datetime(["2026-09-23", "2026-09-24"])
})

dates_df.to_json(
    "dates.json",
    orient="records",
    date_format="iso",
    indent=2
)

print(open("dates.json", encoding="utf-8").read())


# 21. `date_format`

Common options when writing JSON:

### `date_format="epoch"`
Dates become Unix timestamps.

### `date_format="iso"`
Dates become ISO-style strings.

Use ISO when readability and interoperability are important.


In [ ]:
print("ISO:")
print(dates_df.to_json(orient="records", date_format="iso"))

print("\nEpoch:")
print(dates_df.to_json(orient="records", date_format="epoch"))


# 22. `date_unit`

Controls the unit for epoch timestamps.

Common values:

- `s` → seconds
- `ms` → milliseconds
- `us` → microseconds
- `ns` → nanoseconds

Example:


In [ ]:
print(dates_df.to_json(
    orient="records",
    date_format="epoch",
    date_unit="s"
))

print(dates_df.to_json(
    orient="records",
    date_format="epoch",
    date_unit="ms"
))


# 23. Unicode JSON

If your data contains Hindi or other non-ASCII characters, use:

```python
force_ascii=False
```

This keeps characters readable instead of escaping them.


In [ ]:
unicode_df = pd.DataFrame({
    "name": ["Amit", "अमित"],
    "city": ["Noida", "लखनऊ"]
})

print(
    unicode_df.to_json(
        orient="records",
        force_ascii=False,
        indent=2
    )
)


# 24. JSON Lines writing

For large datasets:

```python
df.to_json(
    "data.jsonl",
    orient="records",
    lines=True
)
```

Each row becomes one JSON object.

This is particularly useful for:
- logs
- machine learning datasets
- large API exports
- streaming pipelines


In [ ]:
large_example = pd.DataFrame({
    "id": range(1, 6),
    "value": [10, 20, 30, 40, 50]
})

large_example.to_json(
    "data.jsonl",
    orient="records",
    lines=True
)

print(open("data.jsonl", encoding="utf-8").read())


# 25. Important `read_json()` parameters

### `orient`
Tells Pandas the JSON structure.

### `typ`
Usually:
- `"frame"` → DataFrame
- `"series"` → Series

### `lines`
Set `True` for JSON Lines / NDJSON.

### `dtype`
Controls dtype inference/conversion.

### `convert_dates`
Controls date conversion behavior.

### `encoding`
Useful when reading files with a specific encoding.

### `compression`
Useful for compressed files such as gzip.

### `nrows`
Read only a limited number of rows when supported by the JSON format.

**Rule:** You do not need to memorize every parameter. Learn the common ones and look up specialized parameters when needed.


# 26. Reading only part of a JSON Lines file

For JSON Lines, `lines=True` is essential.

When exploring a huge JSONL file, `nrows` can be useful:

```python
pd.read_json("huge.jsonl", lines=True, nrows=100)
```

This is useful for quickly inspecting a large dataset before loading everything.


In [ ]:
sample_100 = pd.read_json(
    "data.jsonl",
    lines=True,
    nrows=3
)

sample_100


# 27. Working with API responses

A common workflow is:

```text
API
 ↓
response.json()
 ↓
Python list/dict
 ↓
pd.json_normalize()
 ↓
DataFrame
 ↓
clean / analyze / visualize
```

Example with a Python object that looks like an API response:


In [ ]:
api_response = {
    "status": "success",
    "data": [
        {"id": 1, "name": "Amit", "score": 90},
        {"id": 2, "name": "Rahul", "score": 85}
    ]
}

api_df = pd.json_normalize(api_response["data"])
api_df


# 28. API response with metadata + records

When the response contains both metadata and a list of records:

```python
{
    "page": 1,
    "total": 100,
    "results": [...]
}
```

You normally extract the list first:

```python
df = pd.json_normalize(response["results"])
```

Then separately keep metadata if needed.


In [ ]:
api_response = {
    "page": 1,
    "total": 2,
    "results": [
        {"id": 1, "name": "Amit"},
        {"id": 2, "name": "Rahul"}
    ]
}

api_df = pd.json_normalize(api_response["results"])

metadata = {
    "page": api_response["page"],
    "total": api_response["total"]
}

display(api_df)
print("Metadata:", metadata)


# 29. Flattening only selected nested fields

Sometimes complete flattening creates too many columns.

You can select what you need after normalization, or control depth using `max_level`.

Example:


In [ ]:
selected = pd.json_normalize(
    nested_data,
    sep="_",
    max_level=1
)

selected


# 30. `json_normalize()` with errors

When metadata is missing from some records, `errors="ignore"` can prevent an error.

Default:

```python
errors="raise"
```

Alternative:

```python
errors="ignore"
```

Use `ignore` when missing metadata is acceptable and you want the pipeline to continue.


In [ ]:
incomplete = [
    {"id": 1, "profile": {"city": "Noida"}},
    {"id": 2}
]

pd.json_normalize(
    incomplete,
    meta=["profile"],
    errors="ignore"
)


# 31. Practical workflow — messy nested JSON

When you receive an unfamiliar JSON file:

### Step 1
Inspect it with Python:

```python
with open("file.json", encoding="utf-8") as f:
    data = json.load(f)
```

### Step 2
Check the type:

```python
type(data)
```

### Step 3
Look at a small sample:

```python
print(data[:2])
```

or:

```python
print(data)
```

### Step 4
If it is tabular → try `pd.DataFrame()` or `pd.read_json()`.

### Step 5
If it is nested → use `pd.json_normalize()`.

### Step 6
If it contains repeated lists → use `record_path`.

### Step 7
Bring parent fields using `meta`.

### Step 8
Clean the resulting DataFrame normally.


In [ ]:
# A reusable inspection pattern
def inspect_json(data):
    print("Python type:", type(data))
    if isinstance(data, list):
        print("Number of records:", len(data))
        if data:
            print("First record:")
            print(data[0])
    elif isinstance(data, dict):
        print("Top-level keys:", list(data.keys()))

with open("students.json", encoding="utf-8") as f:
    data = json.load(f)

inspect_json(data)


# 32. Common mistakes

### Mistake 1 — Confusing JSON string with Python object

```python
'{"name": "Amit"}'
```

is a **string**.

```python
{"name": "Amit"}
```

is a **Python dictionary**.

Use:

```python
json.loads()
```

to convert JSON text into a Python object.

---

### Mistake 2 — Using `read_json()` for every nested API response

For deeply nested API responses, inspect the structure first. `json_normalize()` is often the better tool.

---

### Mistake 3 — Forgetting `lines=True`

JSON Lines needs:

```python
pd.read_json(file, lines=True)
```

---

### Mistake 4 — Losing parent information

If using `record_path`, add:

```python
meta=[...]
```

when you need parent fields.

---

### Mistake 5 — Assuming all JSON is tabular

JSON can be a dictionary, list, nested object, or deeply nested API response. First understand its structure.


# 33. `DataFrame` → JSON quick reference

| Goal | Code |
|---|---|
| Default JSON | `df.to_json("file.json")` |
| One object per row | `df.to_json("file.json", orient="records")` |
| Pretty JSON | `df.to_json(..., indent=2)` |
| JSON Lines | `df.to_json("file.jsonl", orient="records", lines=True)` |
| Only values | `df.to_json(..., orient="values")` |
| Include index structure | `df.to_json(..., orient="index")` |
| Split structure | `df.to_json(..., orient="split")` |
| Schema + data | `df.to_json(..., orient="table")` |
| Preserve Unicode | `force_ascii=False` |
| ISO dates | `date_format="iso"` |


# 34. JSON → DataFrame quick reference

| Situation | Code |
|---|---|
| Normal JSON file | `pd.read_json("file.json")` |
| JSON Lines | `pd.read_json("file.jsonl", lines=True)` |
| JSON string | `pd.read_json(StringIO(text), orient="records")` |
| Nested list/dict | `pd.json_normalize(data)` |
| Nested list as rows | `pd.json_normalize(data, record_path="items")` |
| Add parent fields | `meta=["id", "name"]` |
| Nested record path | `record_path=["a", "b", "items"]` |
| Limit flattening | `max_level=1` |
| Change nested separator | `sep="_"` |


# 35. Function-by-function cheat sheet

## Core functions

### `pd.read_json()`
**JSON → DataFrame**

Use when the JSON is already reasonably tabular.

### `df.to_json()`
**DataFrame → JSON**

Use when exporting DataFrame data.

### `pd.json_normalize()`
**Nested JSON → flat DataFrame**

Use for API responses and nested dictionaries/lists.

## Python JSON functions

### `json.load()`
JSON **file → Python object**

### `json.loads()`
JSON **string → Python object**

### `json.dump()`
Python object **→ JSON file**

### `json.dumps()`
Python object **→ JSON string**

## Very useful supporting functions

### `explode()`
List in a column → multiple rows.

### `pd.to_datetime()`
Date strings → datetime.

### `astype()`
Change data type.

### `isna()` / `fillna()`
Handle missing data.

### `dropna()`
Remove missing data.


# 36. Interview / practical questions

### Q1. `read_json()` vs `json_normalize()`?

**`read_json()`** is primarily for reading JSON into Pandas when the structure maps reasonably to a DataFrame.

**`json_normalize()`** is specifically useful for flattening nested JSON structures.

---

### Q2. Why use `record_path`?

Because nested arrays/lists often represent repeated records that should become DataFrame rows.

---

### Q3. Why use `meta`?

To copy parent-level information onto each generated child row.

---

### Q4. What is JSON Lines?

A format where each line is an independent JSON object.

Use:

```python
lines=True
```

---

### Q5. `json.loads()` vs `json.load()`?

- `loads` → **string**
- `load` → **file**

Similarly:

- `dumps` → **string**
- `dump` → **file**


# 37. Mini project — JSON → clean DataFrame

We will:

1. Load nested JSON
2. Normalize it
3. Convert dates
4. Clean column names
5. Check missing values
6. Export clean JSON


In [ ]:
project_data = [
    {
        "user_id": 101,
        "profile": {
            "name": "Amit",
            "city": "Noida"
        },
        "joined": "2026-09-01",
        "orders": [
            {"product": "Laptop", "amount": 60000},
            {"product": "Mouse", "amount": 1000}
        ]
    },
    {
        "user_id": 102,
        "profile": {
            "name": "Rahul",
            "city": "Delhi"
        },
        "joined": "2026-09-05",
        "orders": [
            {"product": "Keyboard", "amount": 2000}
        ]
    }
]

orders = pd.json_normalize(
    project_data,
    record_path="orders",
    meta=["user_id", ["profile", "name"], ["profile", "city"], "joined"],
    sep="_"
)

orders["joined"] = pd.to_datetime(orders["joined"])

orders


In [ ]:
# Clean column names
orders.columns = (
    orders.columns
    .str.lower()
    .str.replace(" ", "_")
)

# Check missing values
print(orders.isna().sum())

# Export
orders.to_json(
    "clean_orders.json",
    orient="records",
    date_format="iso",
    indent=2
)

orders


# 38. Final mental model

Think about JSON work like this:

```text
                  JSON
                   |
          +--------+--------+
          |                 |
      Tabular?           Nested?
          |                 |
   pd.read_json()    pd.json_normalize()
          |                 |
          +--------+--------+
                   |
              DataFrame
                   |
          Clean / Analyze
                   |
             df.to_json()
                   |
                  JSON
```

### The 5 functions you should remember first

```python
pd.read_json()
df.to_json()
pd.json_normalize()
json.load()
json.loads()
```

Then learn:

```python
json.dump()
json.dumps()
explode()
pd.to_datetime()
```

**Most important practical skill:** Don't memorize every parameter. First inspect the JSON structure, then choose the function/orientation that matches that structure.


<!-- EXPERIMENT -->
### Experiment: flattening depth
Re-run `pd.json_normalize(nested_data)` with `max_level=0` and then `max_level=1`. Predict first: which one leaves the `address` column as a dict, and which splits it into `address_state`/`address_pincode`? Then try `sep="."` instead of `sep="_"` and watch the column names change. Nothing else in the notebook needs to change.

<!-- DOC-BANNER -->
## Key takeaways (Day 16 deep dive)
- `pd.read_json` is for JSON that already looks tabular; nested lists/dicts need `pd.json_normalize`.
- `orient` must match on read/write (`records`/`columns`/`index`/`split`/`table`/`values`); JSON Lines needs `lines=True` on both sides.
- `record_path` + `meta` flatten nested arrays while keeping parent fields; `sep`/`max_level` control column names/depth.
- `StringIO` wraps a JSON *string* so `read_json` treats it as a file-like buffer.
- **ML connection:** JSON APIs/SQL dumps arrive nested — normalize → `to_datetime` → clean → `isna` checks before any train/test split (see mini-project `clean_orders.json` at the end).
- **Common mistakes:** `read_json` on a raw string (needs `StringIO`); forgetting `lines=True` for JSONL; using `read_json` for deeply nested API payloads instead of `normalize`.
